# Python string primer

A practical reference covering string formatting, prefixes, and f-strings.

## The `%` operator on strings

Printf-style string formatting — the same conventions as C's `printf`, baked into the string type.

In [ ]:
"Hello, %s! You are %d years old." % ("Alice", 30)
# 'Hello, Alice! You are 30 years old.'

The left operand is a format string with `%`-prefixed conversion specifiers; the right operand is the value (or tuple of values) to interpolate.

### Common specifiers

- `%s` calls `str()` on the value
- `%r` calls `repr()`
- `%d` is integer
- `%f` is float
- `%x` is lowercase hex
- `%e` is scientific notation
- `%%` is a literal percent sign

### Width, precision, flags

In [ ]:
"%5d"    % 42       # '   42'      (width 5, right-aligned)
"%-5d|"  % 42       # '42   |'     (left-aligned)
"%05d"   % 42       # '00042'      (zero-padded)
"%.2f"   % 3.14159  # '3.14'       (2 decimal places)
"%8.2f"  % 3.14159  # '    3.14'   (width 8, precision 2)
"%+d"    % 42       # '+42'        (force sign)

### Tuples vs. dicts

Multiple values go in a tuple. A single value can be passed bare _unless_ it's a tuple itself (in which case wrap it):

In [ ]:
"%s and %s" % ("a", "b")
"%s"        % (some_tuple,)   # the comma matters

You can also use named references with a dict:

In [ ]:
"%(name)s is %(age)d" % {"name": "Alice", "age": 30}

### Gotcha

`"%d" % 3.7` truncates to `3` silently. Mismatched argument counts raise `TypeError`.

### Status in modern Python

Still works, not deprecated, but f-strings and `str.format` are preferred for new code. You'll still see `%`-formatting in older codebases, in logging (`logger.info("user %s did %s", user, action)` defers formatting until needed), and anywhere C-style format strings are idiomatic.

## String prefixes

Python supports several prefixes that change how a string literal is interpreted.

### `f"..."` — formatted string literal

Evaluates `{...}` expressions inline. See the dedicated section below.

### `r"..."` — raw string

Backslashes are literal, not escape sequences. Indispensable for regex and Windows paths:

In [ ]:
r"\d+\.\d+"          # regex pattern, no need to write \\d+\\.\\d+
r"C:\Users\Alice"    # path, no escape headaches

One quirk: a raw string still can't _end_ in an odd number of backslashes (`r"\"` is a syntax error).

### `b"..."` — bytes literal

Produces a `bytes` object, not `str`. ASCII-only inside the literal; non-ASCII needs `\xNN` escapes. Used for binary data, network protocols, file headers:

In [ ]:
b"GET / HTTP/1.1\r\n"

### `u"..."` — unicode literal

A no-op in Python 3 — all strings are already unicode. It only exists so Python 2/3 compatible code can write `u"..."` without a syntax error. You'll see it in old codebases; don't write new code with it.

### Combined prefixes

Prefixes can be combined: `rb"..."` is a raw bytes literal (great for regex over bytes), `rf"..."` is a raw f-string (great for regex patterns that interpolate). Order doesn't matter (`br` == `rb`), and case doesn't matter (`Rb` is fine).

### `t"..."` — template string (Python 3.14+, October 2025)

A newer addition. Looks like an f-string but doesn't immediately produce a `str` — it produces a `Template` object that exposes the raw template parts and the interpolated values separately. The point is _deferred or sanitised_ formatting: the receiving function decides how to render the values. Useful for things like SQL or HTML where you want interpolation syntax but need to escape values to prevent injection.

## F-strings in depth

F-strings (PEP 498, Python 3.6+) are the modern way to format strings. The full syntax inside the braces is `{expression:format_spec}`.

### Basic substitution

In [ ]:
name = "Alice"
age = 30
s = f"Hello, {name}! You are {age} years old."

The `f` prefix makes it an f-string, and any expression inside `{...}` gets evaluated and interpolated. No type specifiers needed — `{age}` works whether `age` is an int, float, or string.

### Indexing and attribute access

In [ ]:
p = ("Alice", 30)
s = f"{p[0]} is {p[1]} years old"
# 'Alice is 30 years old'

d = {"name": "Alice", "age": 30}
s = f"{d['name']} is {d['age']} years old"

Before Python 3.12, you couldn't reuse the _same_ quote style inside the braces as outside. In 3.12+ that restriction is gone, so `f"{d["name"]}"` is legal too.

### Expressions, not just names

Anything that's a valid Python expression works inside the braces:

In [ ]:
f"{2 + 2}"                    # '4'
f"{name.upper()}"             # method calls
f"{user['email']}"            # subscripting
f"{[x*2 for x in range(3)]}"  # comprehensions
f"{func(a, b)}"               # function calls

### The format spec

After a colon, you get the same mini-language `format()` and `str.format` use:

```
{value:[fill][align][sign][#][0][width][,_][.precision][type]}
```

In [ ]:
f"{42:5d}"        # '   42'      width 5
f"{42:<5}"        # '42   '      left-align
f"{42:>5}"        # '   42'      right-align (default for numbers)
f"{42:^5}"        # ' 42  '      center
f"{42:*^5}"       # '*42**'      center, fill with *
f"{42:05d}"       # '00042'      zero-pad
f"{1234567:,}"    # '1,234,567'  thousands separator
f"{1234567:_}"    # '1_234_567'  underscore separator
f"{42:+d}"        # '+42'        force sign
f"{-42: d}"       # '-42'        space for positive, minus for negative

### Numeric types

In [ ]:
f"{3.14159:.2f}"   # '3.14'        fixed-point, 2 decimals
f"{3.14159:.3e}"   # '3.142e+00'   scientific
f"{0.875:.1%}"     # '87.5%'       percentage (multiplies by 100)
f"{255:b}"         # '11111111'    binary
f"{255:o}"         # '377'         octal
f"{255:x}"         # 'ff'          hex lowercase
f"{255:X}"         # 'FF'          hex uppercase
f"{255:#x}"        # '0xff'        with prefix
f"{1000000:.2e}"   # '1.00e+06'    scientific with precision

### Strings

For strings, `precision` is a _max length_, not decimal places:

In [ ]:
f"{'hello':.3}"    # 'hel'        truncate to 3 chars
f"{'hi':10}"       # 'hi        '  pad to width 10

### Conversion flags

`!s`, `!r`, `!a` apply `str()`, `repr()`, or `ascii()` before formatting:

In [ ]:
f"{value!r}"       # uses repr() — useful for debug output
f"{value!s:>10}"   # str() then right-align in 10

### The `=` debug specifier (3.8+)

Genuinely useful and underused. It prints both the expression and its value:

In [ ]:
x = 42
f"{x=}"            # 'x=42'
f"{x*2=}"          # 'x*2=84'
f"{x=:.2f}"        # 'x=42.00'   formatting still works

Indispensable for print-debugging.

### Nested format specs

The format spec itself can contain `{}`:

In [ ]:
width = 10
precision = 3
f"{3.14159:{width}.{precision}f}"   # '     3.142'

This is how you parameterize formatting at runtime.

### Dates

Datetime objects respect their own format codes inside the spec:

In [ ]:
from datetime import datetime
now = datetime.now()
f"{now:%Y-%m-%d}"               # '2026-05-04'
f"{now:%B %d, %Y at %I:%M %p}"  # 'May 04, 2026 at 02:30 PM'

Anything that defines `__format__` plays in this system; datetime is just the most common.

### Multi-line and triple-quoted

In [ ]:
msg = f"""
Name:  {name}
Email: {email}
Score: {score:.2f}
"""

### Gotchas

Backslashes can't appear inside the braces in older Python (pre-3.12). So `f"{'\n'.join(items)}"` was a syntax error — you had to assign `nl = '\n'` first. In 3.12+ this restriction is lifted along with the matching-quotes one.

Curly braces themselves are escaped by doubling: `f"{{literal braces}}"` produces `{literal braces}`.

F-strings are evaluated immediately at the line where they appear — they capture values then, not later. If you want a "template" you fill in repeatedly, use `str.format` or the new `t"..."` template strings.

## String operations

Strings are immutable sequences of Unicode code points. Every operation that "modifies" a string returns a new one; the original is unchanged.

### Indexing and slicing

The same machinery as lists — zero-based, negative indices count from the end, slices are `[start:stop:step]`.

In [ ]:
s = "Hello, World!"
s[0]              # 'H'
s[-1]             # '!'
s[7:12]           # 'World'
s[:5]             # 'Hello'
s[::-1]           # '!dlroW ,olleH'   reversed
len(s)            # 13

Indexing a string by integer returns a one-character string, not a separate `char` type — Python has no character type.

### Concatenation and repetition

In [ ]:
"Hello" + ", " + "World"        # 'Hello, World'
"-" * 20                        # '--------------------'
"".join(["a", "b", "c"])        # 'abc'
", ".join(["a", "b", "c"])      # 'a, b, c'

`+` is fine for a handful of strings. For many strings in a loop, prefer `"".join(parts)` — it allocates once instead of once per concatenation.

### Case

In [ ]:
"hello world".upper()           # 'HELLO WORLD'
"HELLO WORLD".lower()           # 'hello world'
"hello world".title()           # 'Hello World'
"hello world".capitalize()      # 'Hello world'   (only first letter)
"Hello World".swapcase()        # 'hELLO wORLD'
"Straße".casefold()             # 'strasse'       aggressive lowercase for case-insensitive compare

`casefold()` is what you want for case-insensitive comparison across languages where `lower()` isn't enough (German `ß`, Turkish dotless I, etc.). For ASCII-only text, `lower()` and `casefold()` are equivalent.

### Searching and counting

In [ ]:
s = "the quick brown fox jumps over the lazy dog"

"quick" in s                    # True            — preferred for membership
s.find("quick")                 # 4               — index, or -1 if not found
s.find("zebra")                 # -1
s.index("quick")                # 4               — like find, but ValueError if missing
s.rfind("the")                  # 31              — rightmost occurrence
s.count("the")                  # 2
s.count("o")                    # 4
s.startswith("the")             # True
s.endswith(("dog", "cat"))      # True            — accepts a tuple of options
s.startswith("quick", 4)        # True            — start position

Use `in` for membership, `find` only when you need the position. `index`/`rindex` raise on missing — pick them when "not found" should propagate as an error.

### Stripping

In [ ]:
"  hello  ".strip()             # 'hello'         both ends, default whitespace
"  hello  ".lstrip()            # 'hello  '
"  hello  ".rstrip()            # '  hello'

",,hello,,".strip(",")          # 'hello'         strip a custom set of characters
"abccba".strip("ab")            # 'cc'

"prefix_value".removeprefix("prefix_")     # 'value'    (3.9+)
"value.txt".removesuffix(".txt")           # 'value'    (3.9+)

`strip(chars)` is **not** a substring removal — the argument is a *set of characters*, any of which gets removed from each end. To remove a literal prefix or suffix, use `removeprefix`/`removesuffix` (3.9+) or slice.

### Replacing

In [ ]:
"hello world".replace("o", "0")            # 'hell0 w0rld'
"hello world".replace("o", "0", 1)         # 'hell0 world'   max 1 replacement
"a.b.c.d".replace(".", "-")                # 'a-b-c-d'

For many simultaneous one-character substitutions, `str.translate` with a translation table is faster:

In [ ]:
table = str.maketrans("aeiou", "AEIOU")
"hello world".translate(table)             # 'hEllO wOrld'

table = str.maketrans("", "", "aeiou")     # delete characters
"hello world".translate(table)             # 'hll wrld'

For pattern-based replacement, see `re.sub` below.

### Splitting and joining

In [ ]:
"a,b,c".split(",")              # ['a', 'b', 'c']
"a,b,c".split(",", 1)           # ['a', 'b,c']           max 1 split
"a b\tc\nd".split()             # ['a', 'b', 'c', 'd']   no arg: collapse runs of whitespace
"a,,b".split(",")               # ['a', '', 'b']         '' for adjacent separators
"a,b,c".rsplit(",", 1)          # ['a,b', 'c']           split from the right

"line1\nline2\nline3".splitlines()
# ['line1', 'line2', 'line3']

"a,b,c".partition(",")          # ('a', ',', 'b,c')      first separator, three-way
"a,b,c".rpartition(",")         # ('a,b', ',', 'c')      last separator
"abc".partition(",")            # ('abc', '', '')        no separator: original + two empties

",".join(["a", "b", "c"])       # 'a,b,c'
"\n".join(lines)                # newline-separated

`split()` with no argument is the right one for whitespace-separated tokens — it ignores leading/trailing whitespace and collapses runs. `split(" ")` is literal and produces empty strings between adjacent spaces.

### Padding and alignment

In [ ]:
"42".ljust(5, "0")              # '42000'
"42".rjust(5, "0")              # '00042'
"42".center(5)                  # ' 42  '
"42".zfill(5)                   # '00042'         like rjust('0'), handles signs
"-7".zfill(5)                   # '-0007'         sign stays at the left

"a\tb\tc".expandtabs(4)         # 'a   b   c'     tabs to spaces

For richer formatting, prefer f-strings (`f"{42:>5}"`, `f"{42:0>5}"`).

### Type tests

In [ ]:
"hello".isalpha()               # True            letters only
"hello123".isalnum()            # True            letters + digits
"42".isdigit()                  # True
"3.14".isdigit()                # False           '.' isn't a digit
" \t\n".isspace()               # True
"Hello".istitle()               # True
"HELLO".isupper()               # True
"hello".islower()               # True
"hello world".isidentifier()    # False           contains a space
"hello_world".isidentifier()    # True
"123".isnumeric()               # True            broader than isdigit (Unicode numerics)
"hello".isascii()               # True

These classify characters; they don't validate "is this a valid number?" — for that, wrap `int(s)` or `float(s)` in `try`/`except`. An empty string returns `False` for all of these.

### Encoding and decoding

In [ ]:
"hello".encode("utf-8")                    # b'hello'
"café".encode("utf-8")                     # b'caf\xc3\xa9'
b"caf\xc3\xa9".decode("utf-8")             # 'café'

"€".encode("ascii")                        # UnicodeEncodeError
"€".encode("ascii", errors="ignore")       # b''
"€".encode("ascii", errors="replace")      # b'?'
"€".encode("ascii", errors="xmlcharrefreplace")    # b'&#8364;'

Default codec is UTF-8 in both directions. Be explicit when reading or writing files: `open(path, encoding="utf-8")`. Latin-1 / cp1252 mistakes are the single most common source of `UnicodeDecodeError` in real codebases.

### Comparison

In [ ]:
"abc" == "abc"                  # True
"abc" < "abd"                   # True            lexicographic by code point
"Abc" < "abc"                   # True            uppercase code points come first
"abc".lower() == "ABC".lower()  # True            simple case-insensitive
"Straße".casefold() == "STRASSE".casefold()    # True   robust case-insensitive

Equality and ordering are character-by-character on Unicode code points, not on locale-aware collation. For real natural-language sorting, use `locale.strxfrm` or `icu`.

## Regular expressions

The standard-library `re` module covers regex search, match, replace, and split. Patterns are written as strings — almost always as raw strings (`r"..."`) so backslashes don't need to be doubled.

### Basic operations

In [ ]:
import re

# Search anywhere in the string
m = re.search(r"\d+", "Order 12345 placed")
m.group()                       # '12345'
m.span()                        # (6, 11)

# Match only at the start of the string
re.match(r"\d+", "Order 12345")     # None       — doesn't start with digits
re.match(r"\w+", "Order 12345")     # match for 'Order'

# Match the whole string
re.fullmatch(r"\d+", "12345")           # match
re.fullmatch(r"\d+", "12345 abc")       # None

# Find all non-overlapping matches as strings
re.findall(r"\d+", "1 + 22 + 333")      # ['1', '22', '333']

# Iterate matches with full Match objects (positions, groups, etc.)
for m in re.finditer(r"\d+", "1 + 22 + 333"):
    print(m.group(), m.span())
# 1 (0, 1)
# 22 (4, 6)
# 333 (9, 12)

# Substitute
re.sub(r"\d+", "N", "1 + 22 + 333")             # 'N + N + N'
re.sub(r"\d+", "N", "1 + 22 + 333", count=2)    # 'N + N + 333'

# Substitute with a function
re.sub(r"\d+", lambda m: str(int(m.group()) * 2), "1 + 22 + 333")
# '2 + 44 + 666'

# Split by regex
re.split(r"[,;]\s*", "a, b; c,d")               # ['a', 'b', 'c', 'd']

`findall` returns strings if there are no groups, single strings if there is one group, and tuples if there are multiple groups — this is a frequent surprise. For uniform behavior, use `finditer`.

### Match objects

A successful match returns a `Match`; no match returns `None`. Always check before calling methods.

In [ ]:
if m := re.search(r"\d+", text):     # walrus pairs nicely
    print(m.group())

m.group()                       # whole match
m.group(0)                      # same as group()
m.group(1)                      # first capturing group
m.groups()                      # tuple of all groups
m.groupdict()                   # dict of named groups
m.span()                        # (start, end)
m.start(); m.end()              # individual positions

### Pattern syntax

The minimum that covers most real cases:

| Pattern         | Meaning                                                |
| --------------- | ------------------------------------------------------ |
| `.`             | Any character except newline (also newline with `re.S`) |
| `\d` `\D`       | Digit / non-digit                                       |
| `\w` `\W`       | Word char (letter, digit, `_`) / non-word char          |
| `\s` `\S`       | Whitespace / non-whitespace                             |
| `\b`            | Word boundary                                           |
| `^` `$`         | Start / end of string (per-line with `re.M`)            |
| `[abc]`         | Any of a, b, c                                          |
| `[^abc]`        | Anything but a, b, c                                    |
| `[a-z]`         | Range                                                   |
| `a*` `a+` `a?`  | 0+, 1+, 0 or 1                                          |
| `a{3}`          | Exactly 3                                               |
| `a{3,5}`        | 3 to 5                                                  |
| `a{3,}`         | 3 or more                                               |
| `a*?` `a+?`     | Non-greedy versions                                     |
| `(...)`         | Capturing group                                         |
| `(?:...)`       | Non-capturing group                                     |
| `(?P<name>...)` | Named group                                             |
| `a|b`           | Alternation                                             |
| `(?=...)`       | Positive lookahead                                      |
| `(?!...)`       | Negative lookahead                                      |
| `(?<=...)`      | Positive lookbehind                                     |
| `(?<!...)`      | Negative lookbehind                                     |

### Groups

Parentheses both group and capture. Capturing groups are numbered from 1 (group 0 is the whole match):

In [ ]:
m = re.search(r"(\d{4})-(\d{2})-(\d{2})", "Date: 2026-05-04")
m.group()                       # '2026-05-04'
m.group(1)                      # '2026'
m.groups()                      # ('2026', '05', '04')

Named groups read better and unpack into a dict:

In [ ]:
m = re.search(r"(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})", "Date: 2026-05-04")
m.group("year")                 # '2026'
m.groupdict()                   # {'year': '2026', 'month': '05', 'day': '04'}

`(?:...)` groups without capturing — useful when you need parens for grouping or alternation but don't want the result to clutter `groups()`:

In [ ]:
re.findall(r"(?:Mr|Ms|Dr)\.\s+(\w+)", "Dr. Smith and Mr. Jones")
# ['Smith', 'Jones']            — only the name, the title is grouped but not captured

In `re.sub`, refer to groups by `\1`, `\2`, ... or `\g<name>`:

In [ ]:
re.sub(r"(\w+) (\w+)", r"\2 \1", "John Smith")               # 'Smith John'
re.sub(r"(?P<y>\d{4})-(?P<m>\d{2})", r"\g<m>/\g<y>", "2026-05")    # '05/2026'

### Flags

Pass `flags=` to any `re` function or embed inline as `(?aiLmsux)` at the start of the pattern. Combine with `|`.

| Flag                     | Effect                                                |
| ------------------------ | ----------------------------------------------------- |
| `re.IGNORECASE` / `re.I` | Case-insensitive matching                             |
| `re.MULTILINE` / `re.M`  | `^` and `$` match at line boundaries                  |
| `re.DOTALL` / `re.S`     | `.` matches newline too                               |
| `re.VERBOSE` / `re.X`    | Allow whitespace and `#` comments inside the pattern  |
| `re.ASCII` / `re.A`      | `\w`, `\d`, `\s` match ASCII only (not Unicode)       |

Verbose mode is the right choice for any non-trivial pattern:

In [ ]:
DATE = re.compile(r"""
    (?P<year>\d{4})     # YYYY
    -
    (?P<month>\d{2})    # MM
    -
    (?P<day>\d{2})      # DD
""", re.VERBOSE)

### Compiled patterns

If the same pattern runs many times, compile it once. The compiled object exposes the same methods:

In [ ]:
DATE = re.compile(r"\d{4}-\d{2}-\d{2}")

for line in lines:
    if DATE.search(line):
        ...

For one-off use, the module-level functions are fine — `re` keeps an internal cache of recently used patterns.

### Greediness

Quantifiers are greedy by default: they match as much as possible. Append `?` for non-greedy (lazy) matching:

In [ ]:
re.search(r"<.+>",  "<a><b>").group()      # '<a><b>'   greedy
re.search(r"<.+?>", "<a><b>").group()      # '<a>'      lazy

For HTML or XML, use a real parser (`html.parser`, `lxml`, `BeautifulSoup`); regex is fragile against nested or malformed markup.

### Common pitfalls

**Forgetting the raw prefix.** Without `r"..."`, `\d` survives string parsing as `\d` only by luck (no `\d` escape exists), but `\b` collides with the backspace escape (`\x08`) and silently means something different. Always use raw strings for patterns.

**Confusing `re.match` with `re.fullmatch`.** `re.match` only requires the *start* of the string to match; the rest can be anything. To validate the whole string, use `re.fullmatch` or anchor with `^...\Z`.

**Calling `.group()` on `None`.** `re.search` and friends return `None` on no match, and `None.group()` raises `AttributeError`. Always check, ideally with the walrus operator.

**Catastrophic backtracking.** Patterns like `(a+)+b` against `"aaaaaaaaaaaaa!"` can take exponential time. If a regex feels slow, suspect it. Avoid nested quantifiers over overlapping character classes.

**Reaching for regex when string methods would do.** For a literal substring, `"abc" in s` is faster and clearer than `re.search(r"abc", s)`. For a literal replacement, `s.replace("a", "b")` beats `re.sub(r"a", "b", s)`. Use regex when the pattern is genuinely a pattern.

**`findall` shape surprise.** Zero groups → list of full matches; one group → list of group-1 strings; multiple groups → list of tuples. Switching from `r"\d+"` to `r"(\d+)"` silently changes the return shape. `finditer` is uniform — every iteration yields a `Match` object.

## Quick reference: which to use

- **New code, normal formatting** — f-strings
- **Logging** — `%`-style with `logger.info("...", arg)` so formatting is deferred
- **Regex patterns** — `r"..."` (or `rf"..."` if interpolating)
- **Windows paths** — `r"..."` or use `pathlib`
- **Binary data** — `b"..."`
- **Reusable templates filled in later** — `str.format` or `t"..."` (3.14+)
- **SQL/HTML with user input** — `t"..."` if available, otherwise a proper escaping library (never raw f-string interpolation)